# KKBOX 표본 고객 50,000명 선정

## 목적

- 전체 학습 고객 중 프로젝트에서 사용할 고객 50,000명을 선정한다.
- 팀원 모두 동일한 고객을 기준으로 EDA와 피처 엔지니어링을 진행한다.
- `msno`를 공통 고객 식별자로 사용한다.
- `is_churn` 비율을 유지하도록 층화 추출한다.

## 표본 추출 기준

- 기준 데이터: `data/raw/train_v2.csv`
- 고객 식별 컬럼: `msno`
- 타깃 컬럼: `is_churn`
- 표본 크기: 50,000명
- 난수 고정값: `random_state=42`

## 프로젝트 예측 기준

- 관찰 종료일: 2017-02-28
- 이탈 예측 기간: 2017-03-01 ~ 2017-03-31

# 1. 라이브러리 및 경로 설정

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
SAMPLE_SIZE = 50_000

# 노트북 위치: 프로젝트/notebooks/
# 부모 폴더인 프로젝트 최상위를 기준으로 경로 설정
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = RAW_DIR / "train_v2.csv"
OUTPUT_PATH = PROCESSED_DIR / "sample_customers_ids_50000.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("현재 실행 위치:", Path.cwd())
print("프로젝트 경로:", PROJECT_ROOT)
print("원본 데이터 경로:", TRAIN_PATH)
print("저장 경로:", OUTPUT_PATH)

현재 실행 위치: c:\dev\project\2차 단위 프로젝트\2nd-feature-data\notebooks
프로젝트 경로: c:\dev\project\2차 단위 프로젝트\2nd-feature-data
원본 데이터 경로: c:\dev\project\2차 단위 프로젝트\2nd-feature-data\data\raw\train_v2.csv
저장 경로: c:\dev\project\2차 단위 프로젝트\2nd-feature-data\data\processed\sample_customers_ids_50000.csv


# 2. 원본 파일 확인

In [2]:
if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"train_v2.csv를 찾을 수 없습니다.\n"
        f"확인 경로: {TRAIN_PATH}"
    )

print("train_v2.csv 확인 완료")

train_v2.csv 확인 완료


# 3. 필요한 컬럼 불러오기

In [3]:
train = pd.read_csv(
    TRAIN_PATH,
    usecols=["msno", "is_churn"]
)

print("데이터 크기:", train.shape)

train.head()

데이터 크기: (970960, 2)


,msno,is_churn
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1


# 4. 기본 구조 확인

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 970960 entries, 0 to 970959
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   msno      970960 non-null  object
 1   is_churn  970960 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 14.8+ MB


In [7]:
print("전체 행 수:", len(train))
print("고유 고객 수:", train["msno"].nunique())
print("고객 ID 결측치:", train["msno"].isna().sum())
print("타깃 결측치:", train["is_churn"].isna().sum())
print("중복 고객 ID:", train["msno"].duplicated().sum())

전체 행 수: 970960
고유 고객 수: 970960
고객 ID 결측치: 0
타깃 결측치: 0
중복 고객 ID: 0


# 5. 타깃 분포 확인

In [8]:
target_summary = pd.DataFrame({
    "count": train["is_churn"].value_counts(dropna=False),
    "ratio": train["is_churn"].value_counts(
        normalize=True,
        dropna=False
    )
})

target_summary

,count,ratio
is_churn,,
0,883630,0.910058
1,87330,0.089942


# 6. 표본 후보 정리

In [9]:
sample_candidates = (
    train[["msno", "is_churn"]]
    .dropna(subset=["msno", "is_churn"])
    .drop_duplicates(subset=["msno"])
    .reset_index(drop=True)
)

print("정리 전 행 수:", len(train))
print("정리 후 후보 고객 수:", len(sample_candidates))
print("고유 고객 수:", sample_candidates["msno"].nunique())

정리 전 행 수: 970960
정리 후 후보 고객 수: 970960
고유 고객 수: 970960


In [11]:
target_counts_by_customer = train.groupby("msno")["is_churn"].nunique()

conflicting_customers = target_counts_by_customer[
    target_counts_by_customer > 1
]

print("타깃값이 서로 다른 중복 고객 수:", len(conflicting_customers))

assert len(conflicting_customers) == 0, (
    "동일한 고객에게 서로 다른 is_churn 값이 존재합니다."
)

타깃값이 서로 다른 중복 고객 수: 0


# 7. 50,000명 추출

In [12]:
if len(sample_candidates) < SAMPLE_SIZE:
    raise ValueError(
        f"표본 후보가 {len(sample_candidates):,}명으로 "
        f"요청한 {SAMPLE_SIZE:,}명보다 적습니다."
    )

sample_customers, _ = train_test_split(
    sample_candidates,
    train_size=SAMPLE_SIZE,
    random_state=RANDOM_STATE,
    stratify=sample_candidates["is_churn"]
)

sample_customers = (
    sample_customers
    .sort_values("msno")
    .reset_index(drop=True)
)

print("표본 추출 완료:", sample_customers.shape)

sample_customers.head()

표본 추출 완료: (50000, 2)


,msno,is_churn
0,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,0
1,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,0
2,++HYwmBh/zxrF/wAgCF/TCntq5vjc6TpCEFWTz6AzaI=,0
3,++boFsOAGvAI3O+P4RG9O+p7e/dF7JdGb6/b+mf0ahk=,1
4,++l8WoNUmsqs7C9ZVyk3pdxXklhdJcSrj50v5su2x/w=,0


# 8. 전체와 표본의 이탈 비율 비교

In [13]:
target_comparison = pd.DataFrame({
    "전체 고객 수": (
        sample_candidates["is_churn"]
        .value_counts()
        .sort_index()
    ),
    "전체 비율": (
        sample_candidates["is_churn"]
        .value_counts(normalize=True)
        .sort_index()
    ),
    "표본 고객 수": (
        sample_customers["is_churn"]
        .value_counts()
        .sort_index()
    ),
    "표본 비율": (
        sample_customers["is_churn"]
        .value_counts(normalize=True)
        .sort_index()
    )
})

target_comparison.index.name = "is_churn"

target_comparison

,전체 고객 수,전체 비율,표본 고객 수,표본 비율
is_churn,,,,
0,883630,0.910058,45503,0.91006
1,87330,0.089942,4497,0.08994


In [14]:
print("표본 행 수:", len(sample_customers))
print("고유 고객 수:", sample_customers["msno"].nunique())
print("고객 ID 결측치:", sample_customers["msno"].isna().sum())
print("중복 고객 ID:", sample_customers["msno"].duplicated().sum())
print("타깃 결측치:", sample_customers["is_churn"].isna().sum())

표본 행 수: 50000
고유 고객 수: 50000
고객 ID 결측치: 0
중복 고객 ID: 0
타깃 결측치: 0


In [15]:
print("표본 행 수:", len(sample_customers))
print("고유 고객 수:", sample_customers["msno"].nunique())
print("고객 ID 결측치:", sample_customers["msno"].isna().sum())
print("중복 고객 ID:", sample_customers["msno"].duplicated().sum())
print("타깃 결측치:", sample_customers["is_churn"].isna().sum())

표본 행 수: 50000
고유 고객 수: 50000
고객 ID 결측치: 0
중복 고객 ID: 0
타깃 결측치: 0


# 9. 공통 고객 ID 저장

In [16]:
sample_customer_ids = sample_customers[["msno"]].copy()

sample_customer_ids.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"저장 완료: {OUTPUT_PATH}")

저장 완료: c:\dev\project\2차 단위 프로젝트\2nd-feature-data\data\processed\sample_customers_ids_50000.csv


In [17]:
sample_customer_ids = sample_customers[["msno"]].copy()

sample_customer_ids.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"저장 완료: {OUTPUT_PATH}")

저장 완료: c:\dev\project\2차 단위 프로젝트\2nd-feature-data\data\processed\sample_customers_ids_50000.csv
